# MVTec AD - DBSCAN Anomaly Detection (Pre-extracted Features)

This notebook loads category-wise `.npz` feature files from `Features/`, trains DBSCAN per category, visualizes performance metrics, and saves a full prediction package.

### Fixes applied:
- **Bug 1 (Cell 1):** `matplotlib` backend set to `Agg` before import so plots save correctly in non-interactive environments.
- **Bug 2 (Cell 3 & 8):** `plt.subplots()` with `squeeze=False` added to ensure `axes` is always 2D — safe `reshape(-1)` regardless of row/column count.
- **Bug 3 (Cell 5):** `evaluate_dbscan` — `X_train` parameter renamed to `X_test_only` to remove misleading signature; `n_train` now tracked separately.
- **Bug 4 (Cell 5):** `find_optimal_threshold` — added guard for all-zero or all-one `y_test` edge case that would crash `roc_curve`.
- **Bug 5 (Cell 7):** `use_reduced` now checks **all** categories, not just the first one, preventing silent `KeyError` on categories that failed PCA.
- **Bug 6 (Cell 7):** `scores` loop variable no longer shadowed by Cell 8's inner `scores`—Cell 8 uses `cat_scores` locally.
- **Bug 7 (All plot cells):** All `plt.show()` calls replaced with `plt.savefig()` + `plt.close()` so every figure is actually saved to disk under `dbscan/plots/`.
- **Bug 8 (Cell 9):** Model `.pkl` now saved inside `OUTPUT_ROOT / 'models'` instead of bare cwd.
- **Bug 9 (Cell 10):** CSV export un-commented and directed to `dbscan/tables/`; a styled summary HTML report is also saved to `dbscan/reports/`.

In [1]:
# ============================================================
# Cell 1: Imports, plotting style, and output folders
# FIX: matplotlib backend must be set BEFORE pyplot import
#      so plt.savefig() works in non-interactive / script mode.
# ============================================================

import os
import math
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns

# FIX: Set non-interactive backend BEFORE importing pyplot
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Central output folder for all DBSCAN notebook artifacts.
OUTPUT_ROOT = Path('dbscan')
PLOTS_DIR   = OUTPUT_ROOT / 'plots'
REPORTS_DIR = OUTPUT_ROOT / 'reports'
MODELS_DIR  = OUTPUT_ROOT / 'models'
TABLES_DIR  = OUTPUT_ROOT / 'tables'

for output_path in [OUTPUT_ROOT, PLOTS_DIR, REPORTS_DIR, MODELS_DIR, TABLES_DIR]:
    output_path.mkdir(parents=True, exist_ok=True)

print('Libraries loaded successfully.')
print(f'All outputs will be saved under: {OUTPUT_ROOT.resolve()}')

Libraries loaded successfully.
All outputs will be saved under: D:\Research\Consultancy\DePaul University\Academics and Co-curricular\Quarter 5 (Winter)\DSC - 445 Machine Learning 1\Group Project\MVec_Anomaly_Detection\dbscan


In [2]:
# ============================================================
# Cell 2: Load all pre-extracted feature files
# ============================================================

features_dir = Path('Features')
assert features_dir.exists(), f"Feature folder not found: {features_dir.resolve()}"

feature_files = sorted(features_dir.glob('*_features.npz'))
assert len(feature_files) > 0, 'No *_features.npz files found in Features/'

def load_category_npz(file_path):
    loaded = np.load(file_path, allow_pickle=True)

    train_features = loaded['train_features']
    test_features  = loaded['test_features']

    train_labels = loaded['train_labels'] if 'train_labels' in loaded else np.zeros(len(train_features), dtype=int)

    if 'test_labels' in loaded:
        test_labels = loaded['test_labels']
    else:
        raise ValueError(f'Missing test_labels in {file_path.name}')

    defect_types = (
        loaded['test_defect_types']
        if 'test_defect_types' in loaded
        else np.array(['unknown'] * len(test_labels))
    )

    return {
        'train_features':    train_features,
        'train_labels':      train_labels,
        'test_features':     test_features,
        'test_labels':       test_labels,
        'test_defect_types': defect_types,
    }

data       = {}
categories = []

for f in feature_files:
    category = f.name.replace('_features.npz', '')
    categories.append(category)
    data[category] = load_category_npz(f)

print(f'Total categories loaded: {len(categories)}')
print('-' * 70)
for category in categories:
    d = data[category]
    print(f"{category.upper():<15} Train: {d['train_features'].shape} | Test: {d['test_features'].shape} | Anomalies in test: {np.sum(d['test_labels']==1)}")

Total categories loaded: 15
----------------------------------------------------------------------
BOTTLE          Train: (209, 512) | Test: (83, 512) | Anomalies in test: 63
CABLE           Train: (224, 512) | Test: (150, 512) | Anomalies in test: 92
CAPSULE         Train: (219, 512) | Test: (132, 512) | Anomalies in test: 109
CARPET          Train: (280, 512) | Test: (117, 512) | Anomalies in test: 89
GRID            Train: (264, 512) | Test: (78, 512) | Anomalies in test: 57
HAZELNUT        Train: (391, 512) | Test: (110, 512) | Anomalies in test: 70
LEATHER         Train: (245, 512) | Test: (124, 512) | Anomalies in test: 92
METAL_NUT       Train: (220, 512) | Test: (115, 512) | Anomalies in test: 93
PILL            Train: (267, 512) | Test: (167, 512) | Anomalies in test: 141
SCREW           Train: (320, 512) | Test: (160, 512) | Anomalies in test: 119
TILE            Train: (230, 512) | Test: (117, 512) | Anomalies in test: 84
TOOTHBRUSH      Train: (60, 512) | Test: (42, 512) | 

In [3]:
# ============================================================
# Cell 3: PCA variance analysis
# FIX: squeeze=False ensures axes is always 2D so reshape(-1)
#      never fails when n_categories <= n_cols (single row).
# FIX: plt.show() -> plt.savefig() + plt.close()
# ============================================================

n_categories = len(categories)
n_cols = 5
n_rows = math.ceil(n_categories / n_cols)

# FIX: squeeze=False guarantees axes.shape == (n_rows, n_cols) always
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows), squeeze=False)
axes = axes.reshape(-1)

pca_info = {}

for idx, category in enumerate(categories):
    train_features = data[category]['train_features']

    scaler       = StandardScaler()
    train_scaled = scaler.fit_transform(train_features)

    pca_full           = PCA().fit(train_scaled)
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    n_components_95    = np.argmax(cumulative_variance >= 0.95) + 1

    pca_info[category] = {
        'n_components_95':   n_components_95,
        'cumulative_variance': cumulative_variance,
    }

    ax = axes[idx]
    ax.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, linewidth=2)
    ax.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
    ax.axvline(x=n_components_95, color='g', linestyle='--', label=f'{n_components_95} comps')
    ax.set_title(f"{category.upper()}\n95% at {n_components_95} comps", fontsize=10)
    ax.set_xlabel('Components', fontsize=8)
    ax.set_ylabel('Cum. variance', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

for idx in range(len(categories), len(axes)):
    axes[idx].axis('off')

plt.suptitle('PCA: Cumulative Explained Variance per Category', fontsize=14, fontweight='bold')
plt.tight_layout()

# FIX: Save figure to disk instead of plt.show() which does nothing with Agg backend
pca_plot_path = PLOTS_DIR / 'pca_variance_analysis.png'
plt.savefig(pca_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'PCA variance plot saved to: {pca_plot_path}')

print('\nPCA Summary (95% threshold)')
print('-' * 70)
print(f"{'Category':<15} {'Original':<10} {'Reduced':<10} {'Reduction %':<12}")
print('-' * 70)
for category in categories:
    n = pca_info[category]['n_components_95']
    reduction_pct = ((data[category]['train_features'].shape[1] - n) / data[category]['train_features'].shape[1]) * 100
    print(f"{category.upper():<15} {data[category]['train_features'].shape[1]:<10} {n:<10} {reduction_pct:>9.1f}%")

PCA variance plot saved to: dbscan\plots\pca_variance_analysis.png

PCA Summary (95% threshold)
----------------------------------------------------------------------
Category        Original   Reduced    Reduction % 
----------------------------------------------------------------------
BOTTLE          512        88              82.8%
CABLE           512        114             77.7%
CAPSULE         512        78              84.8%
CARPET          512        129             74.8%
GRID            512        65              87.3%
HAZELNUT        512        141             72.5%
LEATHER         512        112             78.1%
METAL_NUT       512        100             80.5%
PILL            512        117             77.1%
SCREW           512        79              84.6%
TILE            512        133             74.0%
TOOTHBRUSH      512        35              93.2%
TRANSISTOR      512        99              80.7%
WOOD            512        80              84.4%
ZIPPER          512      

In [4]:
# ============================================================
# Cell 4: Fit scaler + PCA per category and transform train/test
# ============================================================

for category in categories:
    X_train      = data[category]['train_features']
    X_test       = data[category]['test_features']
    n_components = pca_info[category]['n_components_95']

    scaler        = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    pca             = PCA(n_components=n_components)
    X_train_reduced = pca.fit_transform(X_train_scaled)
    X_test_reduced  = pca.transform(X_test_scaled)

    data[category]['scaler']        = scaler
    data[category]['pca']           = pca
    data[category]['train_reduced'] = X_train_reduced
    data[category]['test_reduced']  = X_test_reduced

    retained = np.sum(pca.explained_variance_ratio_) * 100
    print(f"{category.upper()} | Train: {X_train.shape} -> {X_train_reduced.shape} | Test: {X_test.shape} -> {X_test_reduced.shape} | Variance: {retained:.2f}%")

print('\nDimensionality reduction complete.')

BOTTLE | Train: (209, 512) -> (209, 88) | Test: (83, 512) -> (83, 88) | Variance: 95.08%
CABLE | Train: (224, 512) -> (224, 114) | Test: (150, 512) -> (150, 114) | Variance: 95.00%
CAPSULE | Train: (219, 512) -> (219, 78) | Test: (132, 512) -> (132, 78) | Variance: 95.06%
CARPET | Train: (280, 512) -> (280, 129) | Test: (117, 512) -> (117, 129) | Variance: 94.99%
GRID | Train: (264, 512) -> (264, 65) | Test: (78, 512) -> (78, 65) | Variance: 95.03%
HAZELNUT | Train: (391, 512) -> (391, 141) | Test: (110, 512) -> (110, 141) | Variance: 94.97%
LEATHER | Train: (245, 512) -> (245, 112) | Test: (124, 512) -> (124, 112) | Variance: 95.03%
METAL_NUT | Train: (220, 512) -> (220, 100) | Test: (115, 512) -> (115, 100) | Variance: 95.03%
PILL | Train: (267, 512) -> (267, 117) | Test: (167, 512) -> (167, 117) | Variance: 95.00%
SCREW | Train: (320, 512) -> (320, 79) | Test: (160, 512) -> (160, 79) | Variance: 95.01%
TILE | Train: (230, 512) -> (230, 133) | Test: (117, 512) -> (117, 133) | Varianc

In [5]:
# ============================================================
# Cell 5: DBSCAN core functions
# FIX: evaluate_dbscan signature clarified (X_train removed,
#      n_train passed explicitly to avoid confusion).
# FIX: find_optimal_threshold guards against degenerate y_test
#      (all-normal or all-anomaly) that would crash roc_curve.
# ============================================================

def estimate_eps_from_train(X_train, min_samples=5, percentile=95):
    """Estimate epsilon from k-distance distribution on training data."""
    k    = max(2, min_samples)
    nbrs = NearestNeighbors(n_neighbors=k, metric='euclidean').fit(X_train)
    distances, _ = nbrs.kneighbors(X_train)
    k_distances  = distances[:, -1]
    eps          = np.percentile(k_distances, percentile)
    return float(eps), k_distances


def train_dbscan(X_train, eps, min_samples=5):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean', n_jobs=-1)
    dbscan.fit(X_train)

    core_samples = dbscan.components_
    if core_samples is None or len(core_samples) == 0:
        # Fallback: use all training points so inference is always defined.
        core_samples = X_train

    nn_core = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(core_samples)

    model_bundle = {
        'dbscan':        dbscan,
        'nn_core':       nn_core,
        'core_samples':  core_samples,
        'eps':           float(eps),
        'min_samples':   int(min_samples),
        'metric':        'euclidean',
        'n_train':       int(len(X_train)),   # FIX: store n_train inside bundle
    }
    return model_bundle


# FIX: Removed unused X_train parameter; n_train now read from model_bundle
def evaluate_dbscan(model_bundle, X_test, y_test):
    """Score test samples by distance to nearest DBSCAN core point."""
    distances, _ = model_bundle['nn_core'].kneighbors(X_test)
    scores       = distances.flatten()

    eps         = model_bundle['eps']
    predictions = np.where(scores > eps, 1, 0)
    accuracy    = np.mean(predictions == y_test)

    try:
        roc_auc = roc_auc_score(y_test, scores)
    except Exception:
        roc_auc = 0.5

    try:
        avg_precision = average_precision_score(y_test, scores)
    except Exception:
        avg_precision = 0.5

    metrics = {
        'accuracy':      float(accuracy),
        'roc_auc':       float(roc_auc),
        'avg_precision': float(avg_precision),
        'n_train':       model_bundle['n_train'],   # FIX: sourced from bundle
        'n_test':        int(len(X_test)),
        'n_anomalies':   int(np.sum(y_test == 1)),
        'anomaly_ratio': float(np.sum(y_test == 1) / len(y_test)),
    }

    return predictions, scores, metrics


def find_optimal_threshold(scores, y_test, method='youden'):
    # FIX: Guard against degenerate label sets that crash roc_curve / pr_curve
    unique_labels = np.unique(y_test)
    if len(unique_labels) < 2:
        fallback = np.median(scores)
        return np.where(scores >= fallback, 1, 0), float(fallback)

    fpr, tpr, thresholds_roc = roc_curve(y_test, scores)
    precision, recall, thresholds_pr = precision_recall_curve(y_test, scores)

    if method == 'youden':
        j_scores = tpr - fpr
        idx      = np.argmax(j_scores)
        threshold = thresholds_roc[idx]
    elif method == 'f1':
        f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
        idx       = np.argmax(f1_scores)
        threshold = thresholds_pr[idx]
    else:
        diff      = np.abs(precision[:-1] - recall[:-1])
        idx       = np.argmin(diff)
        threshold = thresholds_pr[idx]

    predictions_threshold = np.where(scores >= threshold, 1, 0)
    return predictions_threshold, float(threshold)


print('DBSCAN helper functions are ready.')

DBSCAN helper functions are ready.


In [6]:
# ============================================================
# Cell 6: Visualization function
# FIX: plt.show() -> plt.savefig(path) + plt.close()
# ============================================================

def plot_dbscan_results(y_test, scores, predictions, metrics, category, threshold=None):
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(f'DBSCAN Results - {category.upper()}', fontsize=16, fontweight='bold')

    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

    # 1) Score distribution
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.hist(scores[y_test == 0], bins=30, alpha=0.7, label='Normal',  color='blue', density=True)
    ax1.hist(scores[y_test == 1], bins=30, alpha=0.7, label='Anomaly', color='red',  density=True)
    if threshold is not None:
        ax1.axvline(x=threshold, color='green', linestyle='--', label=f'Threshold ({threshold:.3f})')
    ax1.set_xlabel('Anomaly Score (distance to nearest DBSCAN core point)')
    ax1.set_ylabel('Density')
    ax1.set_title('Anomaly Score Distribution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 2) Confusion matrix
    ax2 = fig.add_subplot(gs[0, 1])
    cm  = confusion_matrix(y_test, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'])
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('Actual')
    ax2.set_title('Confusion Matrix')

    # 3) Text metrics
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.axis('off')
    metrics_text = f"""Performance Metrics:

Accuracy:      {metrics['accuracy']:.3f}
ROC-AUC:       {metrics['roc_auc']:.3f}
Avg Precision: {metrics['avg_precision']:.3f}

Dataset:
Train samples: {metrics['n_train']}
Test samples:  {metrics['n_test']}
Anomalies:     {metrics['n_anomalies']}
"""
    ax3.text(0.1, 0.5, metrics_text, fontsize=12, verticalalignment='center',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # 4) ROC
    ax4 = fig.add_subplot(gs[1, 0])
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax4.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC curve (AUC = {metrics['roc_auc']:.3f})")
    ax4.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    ax4.set_xlim([0.0, 1.0])
    ax4.set_ylim([0.0, 1.05])
    ax4.set_xlabel('False Positive Rate')
    ax4.set_ylabel('True Positive Rate')
    ax4.set_title('ROC Curve')
    ax4.legend(loc='lower right')
    ax4.grid(True, alpha=0.3)

    # 5) Precision-Recall
    ax5 = fig.add_subplot(gs[1, 1])
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ax5.plot(recall, precision, color='green', lw=2, label=f"PR curve (AP = {metrics['avg_precision']:.3f})")
    ax5.set_xlabel('Recall')
    ax5.set_ylabel('Precision')
    ax5.set_title('Precision-Recall Curve')
    ax5.legend(loc='lower left')
    ax5.grid(True, alpha=0.3)

    # 6) Score vs index
    ax6 = fig.add_subplot(gs[1, 2])
    colors = ['blue' if y == 0 else 'red' for y in y_test]
    ax6.scatter(range(len(scores)), scores, c=colors, alpha=0.6, s=20)
    if threshold is not None:
        ax6.axhline(y=threshold, color='green', linestyle='--', label=f'Threshold ({threshold:.3f})')
    ax6.set_xlabel('Sample Index')
    ax6.set_ylabel('Anomaly Score')
    ax6.set_title('Anomaly Scores by Sample')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

    plt.tight_layout()

    # FIX: Save figure to disk; plt.show() is a no-op with the Agg backend
    plot_path = PLOTS_DIR / f'{category}_dbscan_results.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Plot saved -> {plot_path}')

    print('\n' + '=' * 60)
    print(f'Classification Report for {category.upper()}')
    print('=' * 60)
    print(classification_report(y_test, predictions, target_names=['Normal', 'Anomaly']))

In [7]:
# ============================================================
# Cell 7: Main execution - DBSCAN on each category separately
# FIX: use_reduced checks ALL categories, not only the first.
# FIX: evaluate_dbscan call updated (X_train arg removed).
# ============================================================

results     = {}
all_metrics = []

# FIX: Verify every category has PCA-reduced features before relying on them
use_reduced  = all('train_reduced' in data[c] for c in categories)
feature_type = 'PCA-reduced (95% variance)' if use_reduced else 'Raw ResNet-18'

print('\n' + '=' * 72)
print('DBSCAN ANOMALY DETECTION')
print(f'Using features: {feature_type}')
print('=' * 72 + '\n')

for category in categories:
    print(f'Processing: {category.upper()}')

    X_train = data[category]['train_reduced'] if use_reduced else data[category]['train_features']
    X_test  = data[category]['test_reduced']  if use_reduced else data[category]['test_features']
    y_test  = data[category]['test_labels']

    min_samples          = 5
    eps, k_distances     = estimate_eps_from_train(X_train, min_samples=min_samples, percentile=95)
    model_bundle         = train_dbscan(X_train, eps=eps, min_samples=min_samples)

    # FIX: evaluate_dbscan no longer takes X_train
    predictions, cat_scores, metrics = evaluate_dbscan(model_bundle, X_test, y_test)

    predictions_threshold, threshold = find_optimal_threshold(cat_scores, y_test, method='youden')

    metrics_thr             = metrics.copy()
    metrics_thr['accuracy'] = float(np.mean(predictions_threshold == y_test))

    results[category] = {
        'predictions':           predictions,
        'predictions_threshold': predictions_threshold,
        'scores':                cat_scores,   # FIX: use cat_scores (not shadowed)
        'metrics':               metrics_thr,
        'threshold':             threshold,
        'model':                 model_bundle,
        'eps':                   eps,
        'min_samples':           min_samples,
        'k_distances':           k_distances,
    }

    all_metrics.append({
        'category':      category,
        'accuracy':      metrics_thr['accuracy'],
        'roc_auc':       metrics_thr['roc_auc'],
        'avg_precision': metrics_thr['avg_precision'],
        'n_train':       metrics_thr['n_train'],
        'n_test':        metrics_thr['n_test'],
        'n_anomalies':   metrics_thr['n_anomalies'],
        'eps':           eps,
        'min_samples':   min_samples,
    })

    plot_dbscan_results(y_test, cat_scores, predictions_threshold, metrics_thr, category, threshold=threshold)

print('\nAll categories processed successfully.')


DBSCAN ANOMALY DETECTION
Using features: PCA-reduced (95% variance)

Processing: BOTTLE
  Plot saved -> dbscan\plots\bottle_dbscan_results.png

Classification Report for BOTTLE
              precision    recall  f1-score   support

      Normal       1.00      0.95      0.97        20
     Anomaly       0.98      1.00      0.99        63

    accuracy                           0.99        83
   macro avg       0.99      0.97      0.98        83
weighted avg       0.99      0.99      0.99        83

Processing: CABLE
  Plot saved -> dbscan\plots\cable_dbscan_results.png

Classification Report for CABLE
              precision    recall  f1-score   support

      Normal       0.76      0.76      0.76        58
     Anomaly       0.85      0.85      0.85        92

    accuracy                           0.81       150
   macro avg       0.80      0.80      0.80       150
weighted avg       0.81      0.81      0.81       150

Processing: CAPSULE
  Plot saved -> dbscan\plots\capsule_dbscan

In [8]:
# ============================================================
# Cell 8: Score distribution across all categories
# FIX: squeeze=False on subplots + local variable cat_scores
#      to avoid shadowing the outer-scope 'scores' name.
# FIX: plt.show() -> plt.savefig() + plt.close()
# ============================================================

n_categories = len(categories)
n_cols = 5
n_rows = math.ceil(n_categories / n_cols)

# FIX: squeeze=False
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows), squeeze=False)
axes = axes.reshape(-1)

for idx, category in enumerate(categories):
    ax = axes[idx]
    # FIX: use local variable cat_scores to avoid shadowing
    cat_scores = results[category]['scores']
    y_test     = data[category]['test_labels']
    threshold  = results[category]['threshold']

    ax.hist(cat_scores[y_test == 0], bins=30, alpha=0.7, label='Normal',  color='blue', density=True)
    ax.hist(cat_scores[y_test == 1], bins=30, alpha=0.7, label='Anomaly', color='red',  density=True)
    ax.axvline(x=threshold, color='green', linestyle='--', linewidth=1.5)
    ax.set_title(category.upper(), fontsize=10)
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

for idx in range(len(categories), len(axes)):
    axes[idx].axis('off')

plt.suptitle('DBSCAN Anomaly Score Distributions Across Categories', fontsize=16, fontweight='bold')
plt.tight_layout()

# FIX: Save instead of show
dist_plot_path = PLOTS_DIR / 'all_categories_score_distributions.png'
plt.savefig(dist_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Score distributions plot saved to: {dist_plot_path}')

Score distributions plot saved to: dbscan\plots\all_categories_score_distributions.png


In [9]:
# ============================================================
# Cell 9: Save complete DBSCAN prediction package
# FIX: Model saved inside OUTPUT_ROOT/models/ not bare cwd.
# ============================================================

prediction_package = {}
for category in categories:
    prediction_package[category] = {
        'model':       results[category]['model'],
        'pca':         data[category].get('pca'),
        'scaler':      data[category].get('scaler'),
        'threshold':   results[category]['threshold'],
        'eps':         results[category]['eps'],
        'min_samples': results[category]['min_samples'],
        'auc_roc':     results[category]['metrics']['roc_auc'],
    }

# FIX: Save to MODELS_DIR, not bare working directory
model_path = MODELS_DIR / 'mvtec_dbscan_full_prediction_system.pkl'
joblib.dump(prediction_package, model_path)

print(f'DBSCAN prediction system saved to: {model_path}')
print('The file contains DBSCAN artifacts for all categories.')

DBSCAN prediction system saved to: dbscan\models\mvtec_dbscan_full_prediction_system.pkl
The file contains DBSCAN artifacts for all categories.


In [10]:
# ============================================================
# Cell 10: Final metrics table + save CSV and HTML report
# FIX: CSV export un-commented and directed to TABLES_DIR.
# NEW:  Styled HTML summary report saved to REPORTS_DIR.
# ============================================================

df_results = pd.DataFrame(all_metrics)
df_results['Category'] = df_results['category'].str.capitalize()
df_results = df_results[['Category', 'accuracy', 'roc_auc', 'avg_precision',
                          'eps', 'min_samples', 'n_train', 'n_test', 'n_anomalies']]
df_results = df_results.rename(columns={
    'accuracy':      'Accuracy',
    'roc_auc':       'AUC-ROC',
    'avg_precision': 'Average Precision (AP)',
    'eps':           'EPS',
    'min_samples':   'Min Samples',
    'n_train':       'Train Samples',
    'n_test':        'Test Samples',
    'n_anomalies':   'Test Anomalies',
})

df_results = df_results.sort_values(by='AUC-ROC', ascending=False)

print('Detailed DBSCAN Performance Metrics:')
print('=' * 80)
print(df_results.to_string(index=False))

# FIX: CSV export (was commented out)
csv_path = TABLES_DIR / 'mvtec_dbscan_results.csv'
df_results.to_csv(csv_path, index=False)
print(f'\nCSV saved to: {csv_path}')

# NEW: Styled HTML summary report
mean_auc = df_results['AUC-ROC'].mean()
mean_ap  = df_results['Average Precision (AP)'].mean()
mean_acc = df_results['Accuracy'].mean()
best_cat = df_results.iloc[0]['Category']
best_auc = df_results.iloc[0]['AUC-ROC']

html_report = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>DBSCAN Anomaly Detection Report - MVTec AD</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
    h1   {{ color: #2c3e50; }}
    h2   {{ color: #34495e; border-bottom: 2px solid #3498db; padding-bottom: 6px; }}
    .summary-grid {{ display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 30px; }}
    .metric-card  {{ background: white; border-radius: 8px; padding: 20px 30px;
                     box-shadow: 0 2px 6px rgba(0,0,0,.1); min-width: 160px; text-align: center; }}
    .metric-card .val {{ font-size: 2em; font-weight: bold; color: #2980b9; }}
    .metric-card .lbl {{ color: #7f8c8d; font-size: 0.85em; margin-top: 4px; }}
    table {{ width: 100%; border-collapse: collapse; background: white;
             box-shadow: 0 2px 6px rgba(0,0,0,.1); border-radius: 8px; overflow: hidden; }}
    th    {{ background: #2c3e50; color: white; padding: 12px 16px; text-align: left; }}
    td    {{ padding: 10px 16px; border-bottom: 1px solid #ecf0f1; }}
    tr:nth-child(even) td {{ background: #f9f9f9; }}
    tr:hover td {{ background: #eaf4fb; }}
    .good  {{ color: #27ae60; font-weight: bold; }}
    .ok    {{ color: #f39c12; font-weight: bold; }}
    .poor  {{ color: #e74c3c; font-weight: bold; }}
    footer {{ margin-top: 40px; color: #aaa; font-size: 0.85em; }}
  </style>
</head>
<body>
  <h1>DBSCAN Anomaly Detection Report</h1>
  <p><strong>Dataset:</strong> MVTec AD &nbsp;|&nbsp;
     <strong>Features:</strong> ResNet-18 + PCA (95% variance) &nbsp;|&nbsp;
     <strong>Categories:</strong> {len(categories)}</p>

  <h2>Summary</h2>
  <div class="summary-grid">
    <div class="metric-card"><div class="val">{mean_auc:.3f}</div><div class="lbl">Mean AUC-ROC</div></div>
    <div class="metric-card"><div class="val">{mean_ap:.3f}</div><div class="lbl">Mean Avg Precision</div></div>
    <div class="metric-card"><div class="val">{mean_acc:.3f}</div><div class="lbl">Mean Accuracy</div></div>
    <div class="metric-card"><div class="val">{best_cat}</div><div class="lbl">Best Category (AUC {best_auc:.3f})</div></div>
  </div>

  <h2>Per-Category Results (sorted by AUC-ROC)</h2>
  <table>
    <tr>
      <th>Category</th><th>Accuracy</th><th>AUC-ROC</th>
      <th>Avg Precision</th><th>EPS</th><th>Train</th><th>Test</th><th>Anomalies</th>
    </tr>
"""

for _, row in df_results.iterrows():
    auc = row['AUC-ROC']
    cls = 'good' if auc >= 0.8 else ('ok' if auc >= 0.65 else 'poor')
    html_report += f"""
    <tr>
      <td><strong>{row['Category']}</strong></td>
      <td>{row['Accuracy']:.3f}</td>
      <td class="{cls}">{auc:.3f}</td>
      <td>{row['Average Precision (AP)']:.3f}</td>
      <td>{row['EPS']:.4f}</td>
      <td>{int(row['Train Samples'])}</td>
      <td>{int(row['Test Samples'])}</td>
      <td>{int(row['Test Anomalies'])}</td>
    </tr>"""

html_report += """
  </table>
  <footer>Generated by DBScan.ipynb &mdash; MVTec AD anomaly detection pipeline</footer>
</body>
</html>
"""

report_path = REPORTS_DIR / 'dbscan_summary_report.html'
report_path.write_text(html_report, encoding='utf-8')
print(f'HTML report saved to: {report_path}')

print('\n' + '=' * 60)
print('ALL OUTPUTS SAVED')
print('=' * 60)
print(f'  Plots:       {PLOTS_DIR}/')
print(f'  CSV table:   {csv_path}')
print(f'  HTML report: {report_path}')
print(f'  Model .pkl:  {MODELS_DIR}/mvtec_dbscan_full_prediction_system.pkl')

Detailed DBSCAN Performance Metrics:
  Category  Accuracy  AUC-ROC  Average Precision (AP)       EPS  Min Samples  Train Samples  Test Samples  Test Anomalies
    Bottle  0.987952 0.993651                0.997898 28.389490            5            209            83              63
      Tile  0.931624 0.971501                0.990058 30.840240            5            230           117              84
  Hazelnut  0.890909 0.949286                0.976266 26.286158            5            391           110              70
   Leather  0.862903 0.922894                0.975986 31.418858            5            245           124              92
      Wood  0.835443 0.887719                0.967565 27.447615            5            247            79              60
    Zipper  0.781457 0.884454                0.963484 29.131099            5            240           151             119
Transistor  0.830000 0.861250                0.842267 25.816159            5            213           100    